# Markov Decision Processes & Reinforcement Learning
## Blood Inventory Management

This notebook runs and compares five methods:
1. Discounted-reward Policy Iteration
2. Discounted-reward Value Iteration
3. Average-reward Policy Iteration
4. Average-reward Value Iteration
5. Tabular Q-learning

**Model assumptions to verify against the assignment PDF**
- State (inventory) and order quantity are integers in `{0, 1, 2, 3}`.
- Demand is `0`, `2`, or `3`, with probabilities `0.1`, `0.4`, and `0.5`.
- Available inventory is capped at 3: `min(state + order, 3)`.
- End-of-period leftover inventory costs 10 per unit.
- Each unmet unit of demand costs 4.
- Reward is the negative of cost, so maximizing reward is equivalent to minimizing expected cost.

If your assignment defines the inventory update or shortage penalty differently, update the shared model before interpreting results.

## 1. Setup

In [27]:
import sys, subprocess, importlib.util
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# In Google Colab, download the repository files into the current runtime.
# Replace this URL with your repository URL if the files are stored elsewhere.
try:
    from google.colab import files  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)

Python: 3.13.16
NumPy: 2.1.3


## 2. Build and validate the MDP

In [3]:
from mdp_model import (
    STATES, ACTIONS, DEMANDS, DEMAND_PROBS, GAMMA,
    MAX_INVENTORY, HOLDING_COST, SHORTAGE_COST,
    transition_and_reward, build_model, greedy_policy
)

P, r = build_model()
print("P shape:", P.shape, "(state, action, next_state)")
print("r shape:", r.shape, "(state, action)")
print("Demand probabilities sum to:", sum(DEMAND_PROBS))
print("All transition rows sum to 1:", np.allclose(P.sum(axis=2), 1.0))
display(pd.DataFrame(P.reshape(16, 4),
                     index=pd.MultiIndex.from_product([range(4), range(4)], names=["state", "action"]),
                     columns=[f"next_state_{s}" for s in range(4)]))

P shape: (4, 4, 4) (state, action, next_state)
r shape: (4, 4) (state, action)
Demand probabilities sum to: 1.0
All transition rows sum to 1: True


next_state_0  next_state_1  next_state_2  next_state_3
state action                                                        
0     0                1.0           0.0           0.0           0.0
      1                0.9           0.1           0.0           0.0
      2                0.9           0.0           0.1           0.0
      3                0.5           0.4           0.0           0.1
1     0                0.9           0.1           0.0           0.0
      1                0.9           0.0           0.1           0.0
      2                0.5           0.4           0.0           0.1
      3                0.5           0.4           0.0           0.1
2     0                0.9           0.0           0.1           0.0
      1                0.5           0.4           0.0           0.1
      2                0.5           0.4           0.0           0.1
      3                0.5           0.4           0.0           0.1
3     0                0.5           0.4           0.0           0.1
      1                0.5           0.4           0.0           0.1
      2                0.5           0.4           0.0           0.1
      3                0.5           0.4           0.0           0.1

## 3. Inspect expected one-step rewards

In [4]:
reward_table = pd.DataFrame(r, index=[f"state {s}" for s in STATES],
                            columns=[f"order {a}" for a in ACTIONS])
display(reward_table.style.format("{:.2f}"))

,order 0,order 1,order 2,order 3
state 0,-9.20,-6.60,-4.00,-7.00
state 1,-6.60,-4.00,-7.00,-7.00
state 2,-4.00,-7.00,-7.00,-7.00
state 3,-7.00,-7.00,-7.00,-7.00


## 4. Discounted-reward Policy Iteration

In [5]:
from discounted_reward_policy_iteration import policy_iteration as discounted_policy_iteration

discounted_pi, discounted_pi_value = discounted_policy_iteration()
print("Optimal policy (state -> order):", discounted_pi.tolist())
print("State values:", np.round(discounted_pi_value, 4))

Optimal policy (state -> order): [2, 1, 0, 0]
State values: [-20.     -20.     -20.     -23.2609]


## 5. Discounted-reward Value Iteration

In [6]:
from discounted_reward_value_iteration import value_iteration as discounted_value_iteration

discounted_vi, discounted_vi_value, discounted_vi_iterations = discounted_value_iteration()
print("Optimal policy (state -> order):", discounted_vi.tolist())
print("State values:", np.round(discounted_vi_value, 4))
print("Iterations:", discounted_vi_iterations)
print("Policies agree:", np.array_equal(discounted_pi, discounted_vi))

Optimal policy (state -> order): [2, 1, 0, 0]
State values: [-20.     -20.     -20.     -23.2609]
Iterations: 111
Policies agree: True


## 6. Average-reward Policy Iteration

In [7]:
from average_reward_policy_iteration import policy_iteration as average_policy_iteration

average_pi, average_pi_reward, average_pi_bias = average_policy_iteration()
print("Policy (state -> order):", average_pi.tolist())
print("Bias values (relative):", np.round(average_pi_bias, 4))
print("Average reward estimate:", round(float(average_pi_reward), 4))

Policy (state -> order): [2, 1, 0, 0]
Bias values (relative): [ 0.      0.      0.     -3.3333]
Average reward estimate: -4.0


## 7. Average-reward Value Iteration

In [8]:
from average_reward_value_iteration import value_iteration as average_value_iteration

average_vi, average_vi_reward, average_vi_bias, average_vi_iterations = average_value_iteration()
print("Policy (state -> order):", average_vi.tolist())
print("Relative values:", np.round(average_vi_bias, 4))
print("Average reward estimate:", round(float(average_vi_reward), 4))
print("Iterations:", average_vi_iterations)

Policy (state -> order): [2, 1, 0, 0]
Relative values: [ 0.      0.      0.     -3.3333]
Average reward estimate: -4.0
Iterations: 12


## 8. Q-learning

In [34]:
from q_learning import q_learning
import numpy as np
import pandas as pd

q_policy, Q, visits = q_learning(
    episodes=50_000,
    steps_per_episode=30,
    alpha=0.5,
    seed=7,
    return_visits=True,
)

print("Learned policy (state -> order):", q_policy.tolist())

print("\nQ-table:")
display(pd.DataFrame(
    np.round(Q, 3),
    index=[f"state {s}" for s in STATES],
    columns=[f"order {a}" for a in ACTIONS],
))

print("\nVisit counts:")
display(pd.DataFrame(
    visits,
    index=[f"state {s}" for s in STATES],
    columns=[f"order {a}" for a in ACTIONS],
))

Learned policy (state -> order): [2, 1, 0, 2]

Q-table:


,order 0,order 1,order 2,order 3
state 0,-19.669,-17.084,-14.463,-17.687
state 1,-17.078,-14.473,-17.581,-17.792
state 2,-14.321,-17.635,-17.841,-17.622
state 3,-17.473,-17.782,-17.437,-17.739



Visit counts:


,order 0,order 1,order 2,order 3
state 0,287880,288171,287647,286815
state 1,60337,60282,59860,60409
state 2,11773,11978,11846,11982
state 3,15369,15374,15126,15151


## 9. Compare policies

In [35]:
comparison = pd.DataFrame({
    "Discounted Policy Iteration": discounted_pi,
    "Discounted Value Iteration": discounted_vi,
    "Average Policy Iteration": average_pi,
    "Average Value Iteration": average_vi,
    "Q-learning": q_policy,
}, index=[f"Inventory state {s}" for s in STATES])
comparison.index.name = "State"
display(comparison)

,Discounted Policy Iteration,Discounted Value Iteration,Average Policy Iteration,Average Value Iteration,Q-learning
State,,,,,
Inventory state 0,2,2,2,2,2
Inventory state 1,1,1,1,1,1
Inventory state 2,0,0,0,0,0
Inventory state 3,0,0,0,0,2
